In [5]:
import pandas as pd
import numpy as np

In [31]:
# The labels from the Excel sheet
expiries = ['3m', '6m', '1y', '2y', '3y', '5y', '10y']
tenors   = ['1y', '2y', '5y', '10y', '30y']

In [32]:
def melt_matrix(matrix_data, value_name):
    """
    Turns a raw 7x5 numpy array into a DataFrame with Expiry/Tenor columns.
    """
    # Create a DataFrame with the raw data and assign column names (Tenors)
    df = pd.DataFrame(matrix_data, columns=tenors)
    
    # Add the Expiry column (Rows)
    df['Expiry'] = expiries
    
    # "Melt" the DataFrame: Turn the 5 Tenor columns into 1 "Tenor" column
    df_melted = df.melt(id_vars=['Expiry'], var_name='Tenor', value_name=value_name)
    
    return df_melted

In [33]:
raw_df = pd.read_excel("../data/raw/Normal_SABR_1Y10Y_Example for HW.xlsx", sheet_name="Inputs")
raw_df.head(20)

,Unnamed: 0,2026-01-19 00:00:00,Unnamed: 2,1,2,5,10,30,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,Forward Rate,1y,2y,5y,10y,30y,NaN,NaN,Forward Rate,1y,2y,5y,10y,30y
2,NaN,NaN,3m,0.034156,0.03393,0.035673,0.038835,0.04194,NaN,NaN,3m,3.4156,3.393,3.5673,3.8835,4.194
3,NaN,NaN,6m,0.033345,0.033777,0.035842,0.039072,0.041992,NaN,NaN,6m,3.3345,3.3777,3.5842,3.9072,4.1992
4,NaN,NaN,1y,0.033189,0.034062,0.036434,0.039689,0.042166,NaN,NaN,1y,3.3189,3.4062,3.6434,3.9689,4.2166
5,NaN,NaN,2y,0.034962,0.035778,0.038065,0.04104,0.042565,NaN,NaN,2y,3.4962,3.5778,3.8065,4.104,4.2565
6,NaN,NaN,3y,0.03663,0.037383,0.039589,0.042358,0.04288,NaN,NaN,3y,3.663,3.7383,3.9589,4.2358,4.288
7,NaN,NaN,5y,0.039854,0.040586,0.042365,0.044542,0.043248,NaN,NaN,5y,3.9854,4.0586,4.2365,4.4542,4.3248
8,NaN,NaN,10y,0.04644,0.046443,0.047273,0.047215,0.042633,NaN,NaN,10y,4.644,4.6443,4.7273,4.7215,4.2633
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
raw_forwards = raw_df.iloc[2:9, 3:8].values
raw_offsets = raw_df.iloc[20:27, 11:16].values
raw_annuities = raw_df.iloc[11:18, 3:8].values
raw_straddles = raw_df.iloc[20:27, 3:8].values
raw_strangles = raw_df.iloc[29:36, 3:8].values
raw_risk_reversals = raw_df.iloc[38:45, 3:8].values
raw_payers = raw_df.iloc[47:54, 3:8].values
raw_receivers = raw_df.iloc[56:63, 3:8].values

In [42]:
df_fwd = melt_matrix(raw_forwards, "Forward")
df_ann = melt_matrix(raw_annuities, "Annuity")
df_offset = melt_matrix(raw_offsets, "Offset")
df_straddle = melt_matrix(raw_straddles, "Straddle")
df_strangle = melt_matrix(raw_strangles, "Strangle")
df_risk_reversal = melt_matrix(raw_risk_reversals, "RiskReversal")
df_payer = melt_matrix(raw_payers, "Payer")
df_receiver = melt_matrix(raw_receivers, "Receiver")

In [43]:
master_df = df_fwd.copy()
master_df = master_df.merge(df_ann, on=["Expiry", "Tenor"])
master_df = master_df.merge(df_offset, on=["Expiry", "Tenor"])
master_df = master_df.merge(df_straddle, on=["Expiry", "Tenor"])
master_df = master_df.merge(df_strangle, on=["Expiry", "Tenor"])
master_df = master_df.merge(df_risk_reversal, on=["Expiry", "Tenor"])
master_df = master_df.merge(df_payer, on=["Expiry", "Tenor"])
master_df = master_df.merge(df_receiver, on=["Expiry", "Tenor"])

In [44]:
master_df

,Expiry,Tenor,Forward,Annuity,Offset,Straddle,Strangle,RiskReversal,Payer,Receiver
0,3m,1y,0.034156,0.95811,0.0025,20,10,-2,4,6
1,6m,1y,0.033345,0.95062,0.005,31,13,-3,5,8
2,1y,1y,0.033189,0.934966,0.01,50,18,-3,7.5,10.5
3,2y,1y,0.034962,0.904361,0.01,78,42,1,21.5,20.5
4,3y,1y,0.03663,0.964664,0.015,96,46,3,24.5,21.5
5,5y,1y,0.039854,0.816361,0.02,118,56,6,31,25
6,10y,1y,0.04644,0.689481,0.02,135,81,7,44,37
7,3m,2y,0.03393,1.885189,0.0025,46,26,-2,12,14
8,6m,2y,0.033777,1.869398,0.005,70,34,-3,15.5,18.5
9,1y,2y,0.034062,1.837582,0.01,107,42,-4,19,23


In [45]:
master_df['Strike_Payer'] = master_df['Forward'] + master_df['Offset']
master_df['Strike_Receiver'] = master_df['Forward'] - master_df['Offset']

In [46]:
mapping_time = {
    '3m': 0.25,
    '6m': 0.5,
    '1y': 1,
    '2y': 2,
    '3y': 3,
    '5y': 5,
    '10y': 10,
    '30y': 30
}
master_df['T_expiry'] = master_df['Expiry'].map(mapping_time)

In [47]:
master_df.head()

,Expiry,Tenor,Forward,Annuity,Offset,Straddle,Strangle,RiskReversal,Payer,Receiver,Strike_Payer,Strike_Receiver,T_expiry
0,3m,1y,0.034156,0.95811,0.0025,20,10,-2,4,6,0.036656,0.031656,0.25
1,6m,1y,0.033345,0.95062,0.005,31,13,-3,5,8,0.038345,0.028345,0.50
2,1y,1y,0.033189,0.934966,0.01,50,18,-3,7.5,10.5,0.043189,0.023189,1.00
3,2y,1y,0.034962,0.904361,0.01,78,42,1,21.5,20.5,0.044962,0.024962,2.00
4,3y,1y,0.03663,0.964664,0.015,96,46,3,24.5,21.5,0.05163,0.02163,3.00


In [49]:
master_df.to_csv("../data/processed/swaption_data_clean.csv", index=False)